<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [6]:
import pandas as pd
PATH = "/content/DS2002FA26/notebooks/06-midterm-walmart/data/product_catalog.csv"
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [4]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_tarts = cat[cat['product_name'].str.contains('pop', case = False, na = False)]

print("Number of pop-tarts rows: ", len(pop_tarts)
)

display(pop_tarts)

Number of pop-tarts rows:  4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [7]:
# TODO
strawberry = pop_tarts[
    pop_tarts['product_name'].str.contains(
        'strawberry', case=False, na=False
    )
]

other_pop_tarts = pop_tarts[
    ~pop_tarts['product_name'].str.contains(
        'strawberry', case=False, na=False
    )
]

print("Strawberry Pop-Tart variants:")
display(strawberry[['sku', 'product_name']])

print("Other Pop-Tart products:")
display(other_pop_tarts[['sku', 'product_name']])

Strawberry Pop-Tart variants:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Other Pop-Tart products:


,sku,product_name


**Pop-Tart rows that are a different product:** _must remain separate because they represent different products. Only SKUs representing the same strawberry Pop-Tart product should be consolidated._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [12]:
original_row_count = len(cat)
canonical_sku = (
    'PT-12' if 'PT-12' in strawberry['sku'].values
    else strawberry['sku'].iloc[0]
)

sku_mapping = {
    sku: canonical_sku
    for sku in strawberry['sku']
}

cat['canonical_sku'] = (
    cat['sku'].map(sku_mapping).fillna(cat['sku'])
)

print("SKU mapping:")
print(sku_mapping)

display(cat[['sku', 'product_name', 'canonical_sku']])

SKU mapping:
{'PT-12': 'PT-12', 'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}


,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,PT-12
1,SKU#459812,Strawberry Pop-Tart,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,PT-12
4,WTR-24,BOTTLED WATER 24PK,WTR-24
5,WATER24PK,Bottled Water 24-Pack,WATER24PK
6,WTR_24PK,Water - 24 Pack,WTR_24PK
7,BATT-100,BATTERIES AA 4PK,BATT-100
8,BAT-AA4,AA Batteries 4-Pack,BAT-AA4
9,BATT_AA_4PK,AA Batteries 4-Pack,BATT_AA_4PK


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [9]:
# TODO
print("Before normalization:")
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = (
    cat['category']
    .str.strip()
    .str.title()
    .replace({'Breakfast Foods': 'Breakfast'})
)

print("\nAfter normalization:")
print(cat['category_clean'].value_counts(dropna=False))

Before normalization:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After normalization:
category_clean
Emergency              9
Entertainment          5
Grocery                5
Breakfast              4
Electronics            4
Household              4
Emergency Supplies     3
Baby                   3
Snacks                 2
Health

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [10]:
import re

def normalize_name(name):
    words = re.findall(r'[a-z0-9]+', str(name).lower())
    return ' '.join(sorted(words))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

groups = (
    cat.groupby('normalized_name')
    .agg(
        sku_count=('sku', 'nunique'),
        skus=('sku', list),
        product_names=('product_name', list)
    )
    .reset_index()
)

duplicates = groups[groups['sku_count'] > 1]

print("Potential fragmented products:")
display(duplicates)

Potential fragmented products:


,normalized_name,sku_count,skus,product_names
2,24 beer light pack,2,"[BEER-24, LTBEER24]","[Light Beer (24-pack), Light Beer (24-pack)]"
8,4 aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]","[AA Batteries 4-Pack, AA Batteries 4-Pack]"
10,aid first kit,2,"[FAID-01, FIRSTAID-KIT]","[First-Aid Kit, First Aid Kit]"
12,board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Monopoly Board Game, Boa..."
13,bread white,2,"[BREAD-WH, WHITEBREAD]","[WHITE BREAD, WHITE BREAD]"
14,butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
15,candles emergency,2,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]"
16,canned soup,2,"[SOUP-CAN, CNSOUP-01]","[Soup - Canned, Canned Soup]"
17,cards playing,2,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]"
20,chips pack variety,2,"[CHIPS-VP, CHIPVARIETY]","[Variety Pack - Chips, Chips Variety Pack]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [13]:
assert cat['canonical_sku'].notna().all()
assert len(cat) == original_row_count

print("All checks passed!")

All checks passed!


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a: Two fragmented product families beyond Pop-Tarts are AA batteries and Monopoly board games. The AA batteries appear under BAT-AA4 and BATT_AA_4PK, while Monopoly appears under BG-MONO, MONOPOLY-01, and GAME-MONO. I would group each family under one canonical SKU after verifying they represent the same product and package size.

b: Consolidating becomes a business decision when two similar products may have different packaging, sizes, or inventory information. For example, strawberry pop-tarts could be sold individually or in multipacks. I would ask the merchandising or inventory team whether these should share a canonical SKU before combining.